In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# YOLOv1 Architecture Implementation
class YOLOv1(nn.Module):
    def __init__(self, S=7, B=2, C=1):
        """
        YOLOv1 architecture
        S: Grid size (7x7)
        B: Number of bounding boxes per cell
        C: Number of classes (1 for face)
        """
        super(YOLOv1, self).__init__()
        self.S = S
        self.B = B
        self.C = C
        
        # Convolutional layers (inspired by original YOLO paper)
        self.conv_layers = nn.Sequential(
            # Layer 1
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Layer 2
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Layer 3-5
            nn.Conv2d(192, 128, kernel_size=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 256, kernel_size=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Layers 6-13
            nn.Conv2d(512, 256, kernel_size=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 256, kernel_size=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 256, kernel_size=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 256, kernel_size=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 512, kernel_size=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 1024, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Layers 14-20
            nn.Conv2d(1024, 512, kernel_size=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 1024, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(1024, 512, kernel_size=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 1024, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(1024, 1024, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(1024, 1024, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.1),
            
            # Layers 21-22
            nn.Conv2d(1024, 1024, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(1024, 1024, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
        )
        
        # Fully connected layers
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024 * S * S, 4096),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.5),
            nn.Linear(4096, S * S * (B * 5 + C))  # 5 = (x, y, w, h, confidence)
        )
    
    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x.reshape(-1, self.S, self.S, self.B * 5 + self.C)

# WIDER FACE Dataset Loader
class WIDERFACEDataset(Dataset):
    def __init__(self, root_dir, annotation_file, transform=None, S=7, B=2):
        self.root_dir = root_dir
        self.transform = transform
        self.S = S
        self.B = B
        self.annotations = self.load_annotations(annotation_file)
    
    def load_annotations(self, anno_file):
        # Load and parse WIDER FACE annotations
        annotations = []
        # Implementation needed based on WIDER FACE format
        return annotations
    
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, idx):
        img_path, boxes = self.annotations[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            image = self.transform(image)
        
        # Convert boxes to YOLO format
        label = self.encode_labels(boxes, image.shape)
        
        return image, label
    
    def encode_labels(self, boxes, img_shape):
        # Encode ground truth boxes into SxS grid format
        S = self.S
        B = self.B
        label = torch.zeros((S, S, B * 5 + 1))
        
        # Implementation of encoding logic
        return label

# YOLO Loss Function
class YOLOLoss(nn.Module):
    def __init__(self, S=7, B=2, C=1, lambda_coord=5, lambda_noobj=0.5):
        super(YOLOLoss, self).__init__()
        self.S = S
        self.B = B
        self.C = C
        self.lambda_coord = lambda_coord
        self.lambda_noobj = lambda_noobj
    
    def forward(self, predictions, targets):
        # Predictions shape: (batch_size, S, S, B*5 + C)
        # Targets shape: (batch_size, S, S, B*5 + C)
        
        # Coordinate loss
        coord_mask = targets[..., 4] > 0  # Object present
        coord_pred = predictions[coord_mask]
        coord_target = targets[coord_mask]
        
        xy_loss = torch.sum((coord_pred[..., :2] - coord_target[..., :2]) ** 2)
        wh_loss = torch.sum((torch.sqrt(coord_pred[..., 2:4]) - torch.sqrt(coord_target[..., 2:4])) ** 2)
        
        # Confidence loss
        conf_loss_obj = torch.sum((coord_pred[..., 4] - coord_target[..., 4]) ** 2)
        
        noobj_mask = targets[..., 4] == 0
        conf_loss_noobj = torch.sum((predictions[noobj_mask][..., 4] - targets[noobj_mask][..., 4]) ** 2)
        
        # Class loss
        class_loss = torch.sum((coord_pred[..., 5:] - coord_target[..., 5:]) ** 2)
        
        # Total loss
        total_loss = (self.lambda_coord * (xy_loss + wh_loss) + 
                     conf_loss_obj + 
                     self.lambda_noobj * conf_loss_noobj + 
                     class_loss)
        
        return total_loss

# Training setup
def train_yolov1(model, train_loader, val_loader, epochs=50, lr=0.001):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
    criterion = YOLOLoss()
    
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0.0
        
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
            if batch_idx % 100 == 0:
                print(f'Epoch [{epoch+1}/{epochs}], Batch [{batch_idx}/{len(train_loader)}], Loss: {loss.item():.4f}')
        
        # Validation
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        
        print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'yolov1_face_best.pth')
        
        scheduler.step()
    
    return model

# Prepare data
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets
train_dataset = WIDERFACEDataset(
    root_dir=f'{OUTPUT_PATH}/images/train',
    annotation_file=f'{OUTPUT_PATH}/train_annotations.txt',
    transform=transform
)

val_dataset = WIDERFACEDataset(
    root_dir=f'{OUTPUT_PATH}/images/val',
    annotation_file=f'{OUTPUT_PATH}/val_annotations.txt',
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4)

# Initialize and train model
yolov1_model = YOLOv1(S=7, B=2, C=1)
print(f"Total parameters: {sum(p.numel() for p in yolov1_model.parameters())}")

# Train the model
trained_model = train_yolov1(yolov1_model, train_loader, val_loader, epochs=50, lr=0.001)

# Evaluate YOLOv1 model
def evaluate_yolov1(model, test_loader):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()
    
    # Implement evaluation metrics (mAP, precision, recall)
    # This requires NMS and IoU calculations
    
    print("YOLOv1 evaluation complete!")

evaluate_yolov1(trained_model, val_loader)
